# 配置
# create new environment, named Lidar2025, add -c conda-forge for my mac
conda create -n Lidar2025 python==3.7  ## 学生版本
conda create -n Lidar2025   ## 我版本，python用了3.9

# activate environment
conda activate Lidar2025 

#add channels 
conda config --add channels https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
conda config --add channels https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/free
conda config --add channels https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/r
conda config --add channels https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/pro
conda config --add channels https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/msys2

#show channel urls
conda config --set show_channel_urls yes

#show channels
conda config --show channels

# Install packages
pip install numpy scikit-learn open3d matplotlib pandas laspy scipy networkx tqdm pandarallel scikit-image ## 学生版本
conda install numpy scikit-learn open3d matplotlib pandas laspy scipy networkx tqdm pandarallel scikit-image -c conda-forge ## 我版本

pip install "urllib3<2.0"## 学生版本
conda install urllib3## 我版本

# 安装kernel
conda install ipykernel 
python -m ipykernel install --user --name lidar2025

In [2]:
## 打开代码后，Tools文件就在同目录下，所以不用设置Tools路径
from Tools.ReadPointCloud import *
import open3d as o3d
from Tools.Treeseg import WatershedSeg
from Tools.Down_Sample import *
from Tools.GroundSegment import *
from Tools.Clean import *
from Tools.Treeseg import *
from Tools.TLS2tree import TLSSeg,MergePoint
import os

In [3]:
#点云数据路径
datapth = r"D:\Desktop\LiDAR_course\1119\Data\FARO_Clip_Resample.las"
spth = r'D:\Desktop\LiDAR_course\1119\Output'

# ##基于laspy读取点云,并转成open3D格式
las = ReadPointCloud(datapth)
pcd = laspy2o3d(las)

Read point cloud: D:\Desktop\LiDAR_course\1119\Data\FARO_Clip_Resample.las


In [3]:
##点云下采样
voxel_size = 0.02
pcd_down_sample = voxel_down_sample(pcd,voxel_size=voxel_size)
o3d.io.write_point_cloud(os.path.join(spth,'Down_sample.pcd'), pcd_down_sample)


Point Cloud Subsample: 0.02


True

In [4]:
#点云去噪
# pcd_radius_outlier, pcd_radius_inlier = radius_outlier_removal(pcd_down_sample,nb_points=6,radius=0.2)
pcd_statistical_outlier, pcd_statistical_inlier = statistical_outlier_removal(pcd_down_sample,neighbors=5,ratio=10.0)
o3d.io.write_point_cloud(os.path.join(spth,'pcd_radius_outlier.pcd'), pcd_statistical_outlier)
o3d.io.write_point_cloud(os.path.join(spth,'pcd_radius_inlier.pcd'), pcd_statistical_inlier)

Set outliers (red) and inliers (gray)
----outlier_cloud:  std::vector<Eigen::Vector3d> with 1496 elements.
Use numpy.asarray() to access data.
----inlier_cloud:  std::vector<Eigen::Vector3d> with 2466629 elements.
Use numpy.asarray() to access data.


True

In [5]:
#地面点分离， 会自动出现可视化，第一个是最低点，关闭后出现第二个试图，最终的点
pcd_statistical_inlier = o3d.io.read_point_cloud(os.path.join(spth,'pcd_radius_inlier.pcd'))
# ground, aboveground = PlaneSegmentation(pcd)
ground, aboveground = PoissonSurfaceFilter(pcd_statistical_inlier,pillar_size=1,threshold=0.3)
o3d.io.write_point_cloud(os.path.join(spth,'ground.pcd'), ground)
o3d.io.write_point_cloud(os.path.join(spth,'aboveground.pcd'), aboveground)

Find the lowest point in pillars
run Poisson surface reconstruction
[Open3D DEBUG] Input Points / Samples: 83 / 83
[Open3D DEBUG] #   Got kernel density: 0.00200009 (s), 548.402 (MB) / 548.402 (MB) / 966 (MB)
[Open3D DEBUG] #     Got normal field: 0.00600004 (s), 549.531 (MB) / 549.531 (MB) / 966 (MB)
[Open3D DEBUG] Point weight / Estimated Area: 1.943646e-02 / 1.613226e+00
[Open3D DEBUG] #       Finalized tree: 0.00899982 (s), 553.535 (MB) / 553.535 (MB) / 966 (MB)
[Open3D DEBUG] #  Set FEM constraints: 0.062 (s), 553.539 (MB) / 553.539 (MB) / 966 (MB)
[Open3D DEBUG] #Set point constraints: 0.00200009 (s), 553.578 (MB) / 553.578 (MB) / 966 (MB)
[Open3D DEBUG] Leaf Nodes / Active Nodes / Ghost Nodes: 80074 / 66888 / 24625
[Open3D DEBUG] Memory Usage: 553.578 MB
[Open3D DEBUG] # Linear system solved: 0.164 (s), 556.273 (MB) / 556.273 (MB) / 966 (MB)
[Open3D DEBUG] Got average: 0.00200009 (s), 556.305 (MB) / 556.305 (MB) / 966 (MB)
[Open3D DEBUG] Iso-Value: 4.941450e-01 = 4.101404e+01 / 

True

In [4]:
##基于Lewos进行枝叶分离
##单木分割

### add this two lines to avoid argparse error in jupyternotebook
import sys
sys.argv = ['']

pcd = MergePoint(groundpth=os.path.join(spth,'ground_0.07.pcd'),leafwoodpth=os.path.join(spth,'leaf_wood_results_0.07.txt'))
TLSSeg(pcd,spth) 


##点云可视化
# o3d.visualization.draw_geometries([ground, aboveground])
# o3d.io.write_point_cloud(spth, data)

Merge Point Cloud ....
---- parameters ----
odir                               D:\Desktop\LiDAR_course\1119\Output
tindex                             None
n_tiles                            3
n_zeros                            3
overlap                            False
slice_thickness                    0.2
find_stems_boundary                [1.5, 2.0]
find_stems_min_radius              0.025
find_stems_min_points              50
graph_edge_length                  1
graph_maximum_cumulative_gap       inf
min_points_per_tree                0
add_leaves                         1
add_leaves_voxel_length            0.5
add_leaves_edge_length             1
save_diameter_class                False
ignore_missing_tiles               False
pandarallel                        False
verbose                            1
Making dtm...

----- skeletonisation started -----


slice data vertically and clustering: 100%|███████████████████████████████████████████| 72/72 [00:00<00:00, 353.65it/s]


fitting convex hulls to clusters
identifying stems...


ValueError: cannot set a frame with no defined index and a scalar